# Build plot-ready and model-ready tables

Extraction is a deliberate boundary. These are the four table shapes most
analyses need, all taken from Scanpy's PBMC3K.

In [1]:
# /// script
# requires-python = ">=3.12"
# dependencies = [
#     "anndata",
#     "annplyr",
#     "numpy",
#     "pandas",
#     "scanpy",
# ]
# ///

In [2]:
import pandas as pd
import scanpy as sc

import annplyr as ap

adata = sc.datasets.pbmc3k_processed()
adata

AnnData object with n_obs × n_vars = 2638 × 1838
    obs: 'n_genes', 'percent_mito', 'n_counts', 'louvain'
    var: 'n_cells'
    uns: 'draw_graph', 'louvain', 'louvain_colors', 'neighbors', 'pca', 'rank_genes_groups'
    obsm: 'X_pca', 'X_tsne', 'X_umap', 'X_draw_graph_fr'
    varm: 'PCs'
    obsp: 'distances', 'connectivities'

## Cell-level long data

One row per cell per gene, with the metadata carried along. That is the shape a
faceted plot wants.

In [3]:
marker_long = adata.ap.to_tidy(
    obs=["louvain"],
    raw=["MS4A1", "CD79A", "NKG7", "LYZ"],
    max_matrix_values=4 * adata.n_obs,
)

print(marker_long.shape)
marker_long.head()

(10552, 4)


,obs_name,feature,value,louvain
0,AAACATACAACCAC-1,MS4A1,0.0,CD4 T cells
1,AAACATACAACCAC-1,CD79A,0.0,CD4 T cells
2,AAACATACAACCAC-1,NKG7,0.0,CD4 T cells
3,AAACATACAACCAC-1,LYZ,0.693147,CD4 T cells
4,AAACATTGAGCTAC-1,MS4A1,1.94591,B cells


A plotting library consumes this directly, without knowing about AnnData:

```python
import seaborn as sns

sns.catplot(
    data=marker_long, x="louvain", y="value", col="feature", kind="violin", sharey=False
)
```

The `value` column is a pandas sparse column, which keeps large exports cheap.
Call `.sparse.to_dense()` if a downstream method needs a plain float column.

In [4]:
marker_long["value"].dtype

Sparse[float32, 0.0]

## Group-level data for cohort figures

When the figure wants one row per group, summarize before leaving AnnData. No
cell-by-gene intermediate is built.

In [5]:
type_summary = adata.ap.summarize(
    obs={"cells": ap.n(), "median_genes": ap.median("n_genes")},
    raw={"mean_MS4A1": ap.mean("MS4A1"), "mean_LYZ": ap.mean("LYZ")},
    by="louvain",
)

type_summary.round(2)

,louvain,cells,median_genes,mean_MS4A1,mean_LYZ
0,CD4 T cells,1144,809.0,0.03,0.43
1,B cells,342,677.0,0.99,0.39
2,CD14+ Monocytes,480,859.0,0.04,3.55
3,NK cells,154,890.0,0.04,0.35
4,CD8 T cells,316,824.5,0.04,0.36
5,FCGR3A+ Monocytes,150,1272.0,0.06,2.24
6,Dendritic cells,37,1544.0,0.06,3.91
7,Megakaryocytes,15,364.0,0.05,0.60


## A wide model table with embedding coordinates

`to_df()` puts metadata, expression, and embeddings side by side, prefixing
each matrix column with its source.

In [6]:
model_frame = adata.ap.to_df(
    obs=["louvain", "n_genes", "percent_mito"],
    raw=["MS4A1", "CD79A"],
    obsm={"X_pca": ["0", "1"]},
    max_matrix_values=4 * adata.n_obs,
)

model_frame.head()

,louvain,n_genes,percent_mito,raw_MS4A1,raw_CD79A,X_pca_0,X_pca_1
index,,,,,,,
AAACATACAACCAC-1,CD4 T cells,781,0.030178,0.0,0.0,5.556233,0.257714
AAACATTGAGCTAC-1,B cells,1352,0.037936,1.94591,1.386294,7.209530,7.481985
AAACATTGATCAGC-1,CD4 T cells,1131,0.008897,0.0,0.0,2.694438,-1.583658
AAACCGTGCTTCCG-1,CD14+ Monocytes,960,0.017431,0.0,0.0,-10.143295,-1.368530
AAACCGTGTATGCG-1,NK cells,522,0.012245,0.0,0.0,-1.112816,-8.152788


## Reshape after extraction

The tidyr-style helpers work on plain DataFrames, so a long table can be
pivoted into a report layout.

In [7]:
report_table = ap.pivot_wider(
    marker_long,
    id_cols=["obs_name", "louvain"],
    names_from="feature",
    values_from="value",
)

report_table.head()

,obs_name,louvain,CD79A,LYZ,MS4A1,NKG7
0,AAACATACAACCAC-1,CD4 T cells,0.0,0.693147,0.0,0.0
1,AAACATTGAGCTAC-1,B cells,1.386294,1.386294,1.94591,0.693147
2,AAACATTGATCAGC-1,CD4 T cells,0.0,1.098612,0.0,0.0
3,AAACCGTGCTTCCG-1,CD14+ Monocytes,0.0,3.218876,0.0,0.693147
4,AAACCGTGTATGCG-1,NK cells,0.0,0.0,0.0,2.484907


## Takeaway

Long for plotting, grouped for cohort figures, wide for models, pivoted for
reports. Every one of them names its sources and bounds its reads, so the cost
of an extraction is visible in the call that performs it.